In [ ]:

# ============================================================
# CELL 1: Installing All Dependencies
# ============================================================
!pip install -q "langchain>=0.3,<1.0" "langchain-core>=0.3,<1.0" "langchain-community>=0.3,<1.0" \
    langchain-google-genai \
    faiss-cpu sentence-transformers pandas numpy plotly opendatasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 458.9/458.9 kB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 66.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 5.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-classic 1.0.1 requires langchain-core<2.0.0,>=1.2.5, but you have langchain-core 0.3.83 which is incompatible.
langchain-classic 1.0.1 requires langchain-text-splitters<2.0.0,>=1.1.0, but you have langchain-text-splitters 0.3.11 which is incompatible.
google-generativeai 0.8.6 requires google-ai-generativelanguage==0.6.15, but you have google-ai-

In [ ]:


# ============================================================
# CELL 2: Imports & API Key
# ============================================================
import os
import pandas as pd
import numpy as np
from pathlib import Path

# LangChain
from langchain_core.tools import tool
from langchain_core.prompts import PromptTemplate
from langchain_core.documents import Document
from langchain.agents import AgentExecutor, create_react_agent
from langchain.memory import ConversationBufferMemory

# Gemini
from langchain_google_genai import ChatGoogleGenerativeAI

# Vector store
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

# Visualization
import plotly.graph_objects as go

# --- Set API Key ---
# Option A: Colab Secrets (recommended)
from google.colab import userdata
try:
    os.environ["GOOGLE_API_KEY"] = userdata.get("GEMINI_API_KEY")
    print("✅ API key loaded from Colab Secrets")
except:
    # Option B: Paste directly
    os.environ["GOOGLE_API_KEY"] = "PASTE_YOUR_KEY_HERE"
    print("⚠️ Using hardcoded API key — switch to Colab Secrets for security")

print("✅ All imports successful")

✅ API key loaded from Colab Secrets
✅ All imports successful


In [ ]:
# ============================================================
# CELL 3: Download Dataset from Kaggle
# ============================================================
# Method: Direct download using opendatasets or manual upload
# Source: https://www.kaggle.com/datasets/martj42/international-football-results-from-1872-to-2017

import urllib.request

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)
DATA_FILE = DATA_DIR / "results.csv"

if not DATA_FILE.exists():
    url = "https://raw.githubusercontent.com/martj42/international_results/master/results.csv"
    print("📥 Downloading dataset from GitHub...")
    urllib.request.urlretrieve(url, DATA_FILE)
    print(f"✅ Downloaded to {DATA_FILE}")
else:
    print("✅ Dataset already cached")

# Verify
df_check = pd.read_csv(DATA_FILE)
print(f"✅ {len(df_check):,} rows loaded — dataset is good")

📥 Downloading dataset from GitHub...
✅ Downloaded to data/results.csv
✅ 49,071 rows loaded — dataset is good


In [ ]:
 # CELL 4: Load & Preprocess Data
# ============================================================
df = pd.read_csv(DATA_FILE, parse_dates=["date"])
print(f"📊 Full dataset: {len(df):,} matches ({df['date'].min().year}–{df['date'].max().year})")

# Filter to World Cup main tournament only
wc_main = df[df["tournament"].str.contains("FIFA World Cup", case=False, na=False)].copy()
wc_main = wc_main[~wc_main["tournament"].str.contains("qualification", case=False, na=False)].copy()

# Add result column
def get_result(row):
    if row["home_score"] > row["away_score"]: return "home_win"
    elif row["home_score"] < row["away_score"]: return "away_win"
    return "draw"

wc_main["result"] = wc_main.apply(get_result, axis=1)

# Text representation for vector DB
wc_main["text"] = wc_main.apply(lambda r: (
    f"On {r['date'].strftime('%B %d, %Y')}, in the {r['tournament']}, "
    f"{r['home_team']} played against {r['away_team']} in {r['city']}, {r['country']}. "
    f"The score was {r['home_team']} {r['home_score']} - {r['away_score']} {r['away_team']}. "
    f"Result: {r['result'].replace('_', ' ')}."
), axis=1)

print(f"⚽ World Cup matches: {len(wc_main):,}")
print(f"   Years: {wc_main['date'].min().year} — {wc_main['date'].max().year}")
print(f"   Teams: {pd.concat([wc_main['home_team'], wc_main['away_team']]).nunique()}")
wc_main.head(3)

📊 Full dataset: 49,071 matches (1872–2026)
⚽ World Cup matches: 964
   Years: 1930 — 2022
   Teams: 82


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,result,text
1486,1930-07-13,Belgium,United States,0,3,FIFA World Cup,Montevideo,Uruguay,True,away_win,"On July 13, 1930, in the FIFA World Cup, Belgi..."
1487,1930-07-13,France,Mexico,4,1,FIFA World Cup,Montevideo,Uruguay,True,home_win,"On July 13, 1930, in the FIFA World Cup, Franc..."
1488,1930-07-14,Brazil,Yugoslavia,1,2,FIFA World Cup,Montevideo,Uruguay,True,away_win,"On July 14, 1930, in the FIFA World Cup, Brazi..."


In [ ]:
# CELL 5: Build FAISS Vector Store
# ============================================================
CACHE_DIR = Path("cache/faiss_index")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

if (CACHE_DIR / "index.faiss").exists():
    print("✅ Loading cached FAISS index...")
    vectorstore = FAISS.load_local(str(CACHE_DIR), embeddings, allow_dangerous_deserialization=True)
else:
    print("🔨 Building FAISS index...")
    documents = [
        Document(
            page_content=row["text"],
            metadata={
                "date": str(row["date"].date()),
                "home_team": row["home_team"],
                "away_team": row["away_team"],
                "home_score": int(row["home_score"]),
                "away_score": int(row["away_score"]),
                "tournament": row["tournament"],
            }
        )
        for _, row in wc_main.iterrows()
    ]
    vectorstore = FAISS.from_documents(documents, embeddings)
    vectorstore.save_local(str(CACHE_DIR))
    print(f"✅ Built & cached ({len(documents)} docs)")

retriever = vectorstore.as_retriever(search_kwargs={"k": 10})
print("✅ Retriever ready")

/tmp/ipython-input-2887/414294091.py:6: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

🔨 Building FAISS index...
✅ Built & cached (964 docs)
✅ Retriever ready


In [ ]:
# CELL 6: Initialize Gemini LLM
# ============================================================
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.3,
    max_output_tokens=2048,
)

# Quick test
resp = llm.invoke("Say 'Hello World Cup!' in one line.")
print(f"✅ Gemini connected: {resp.content}")

✅ Gemini connected: Hello World Cup!


In [ ]:
# CELL 7: Define All 6 Pipeline Tools
# ============================================================

# --- Tool 1: Dataset Discovery ---
@tool
def dataset_discovery_tool(query: str) -> str:
    """Describes the available World Cup dataset: columns, date range, match count, and available teams."""
    teams = sorted(set(wc_main["home_team"]) | set(wc_main["away_team"]))
    return (
        f"Dataset: International Football Results (World Cup main tournament)\n"
        f"Source: Kaggle (martj42/international-football-results)\n"
        f"Matches: {len(wc_main):,}\n"
        f"Date range: {wc_main['date'].min().year} to {wc_main['date'].max().year}\n"
        f"Columns: {list(wc_main.columns)}\n"
        f"Available teams ({len(teams)}): {', '.join(teams[:20])}... and more.\n"
    )

# --- Tool 2: Data Ingestion Stats ---
@tool
def data_ingestion_tool(query: str) -> str:
    """Returns summary statistics about the ingested World Cup dataset including top teams and goal averages."""
    total_apps = (
        wc_main["home_team"].value_counts() + wc_main["away_team"].value_counts()
    ).fillna(0).astype(int).sort_values(ascending=False).head(10)
    avg_goals = wc_main[["home_score", "away_score"]].sum().sum() / len(wc_main)
    return (
        f"Ingested {len(wc_main)} World Cup matches.\n"
        f"Average goals per match: {avg_goals:.2f}\n"
        f"Most frequent teams:\n{total_apps.to_string()}\n"
    )

# --- Tool 3: RAG Retrieval ---
@tool
def retrieval_tool(query: str) -> str:
    """Searches the World Cup vector database for matches relevant to the user's question. Use for historical questions about specific matches, teams, or tournaments."""
    docs = retriever.invoke(query)
    if not docs:
        return "No relevant matches found in the database."
    return "\n".join(f"{i}. {d.page_content}" for i, d in enumerate(docs, 1))

# --- Tool 4: Head-to-Head Reasoning ---
@tool
def reasoning_tool(query: str) -> str:
    """Analyzes head-to-head record between two teams. Computes wins, losses, draws, goal stats, and recent World Cup form."""
    all_teams = set(wc_main["home_team"]) | set(wc_main["away_team"])
    found = [t for t in all_teams if t.lower() in query.lower()]

    if len(found) < 2:
        return f"Could not identify two teams. Found: {found}. Specify two team names clearly."

    t1, t2 = found[0], found[1]
    h2h = wc_main[
        ((wc_main["home_team"] == t1) & (wc_main["away_team"] == t2)) |
        ((wc_main["home_team"] == t2) & (wc_main["away_team"] == t1))
    ].sort_values("date")

    if h2h.empty:
        return f"No head-to-head World Cup matches found between {t1} and {t2}."

    w1 = w2 = d = g1 = g2 = 0
    details = []
    for _, r in h2h.iterrows():
        if r["home_team"] == t1:
            a, b = r["home_score"], r["away_score"]
        else:
            a, b = r["away_score"], r["home_score"]
        g1 += a; g2 += b
        if a > b: w1 += 1
        elif b > a: w2 += 1
        else: d += 1
        details.append(f"  {r['date'].strftime('%Y-%m-%d')}: {r['home_team']} {r['home_score']}-{r['away_score']} {r['away_team']}")

    def recent_form(team, n=5):
        m = wc_main[(wc_main["home_team"] == team) | (wc_main["away_team"] == team)].sort_values("date", ascending=False).head(n)
        res = []
        for _, r in m.iterrows():
            gf = r["home_score"] if r["home_team"] == team else r["away_score"]
            ga = r["away_score"] if r["home_team"] == team else r["home_score"]
            if gf > ga: res.append("W")
            elif gf < ga: res.append("L")
            else: res.append("D")
        return "".join(res)

    return (
        f"HEAD-TO-HEAD: {t1} vs {t2} (World Cup)\n"
        f"Total matches: {len(h2h)}\n"
        f"{t1} wins: {w1} | {t2} wins: {w2} | Draws: {d}\n"
        f"Goals: {t1} {g1} - {g2} {t2}\n"
        f"Recent WC form (last 5): {t1}: {recent_form(t1)} | {t2}: {recent_form(t2)}\n"
        f"Match history:\n" + "\n".join(details)
    )

# --- Tool 5: LLM Synthesis ---
@tool
def llm_synthesis_tool(query: str) -> str:
    """Synthesizes a natural language answer using Gemini from retrieved data context."""
    prompt = PromptTemplate.from_template(
        "You are a World Cup football expert. Based on the following information, "
        "provide a clear, accurate, and engaging answer.\n\n"
        "Information:\n{query}\n\n"
        "Provide your analysis with key facts and any limitations in the data."
    )
    chain = prompt | llm
    return chain.invoke({"query": query}).content

# --- Tool 6: Match Prediction Report ---
@tool
def report_generation_tool(query: str) -> str:
    """Generates a structured match prediction report for two teams. Combines head-to-head data, recent form, and stats."""
    h2h_data = reasoning_tool.invoke(query)
    prompt = PromptTemplate.from_template(
        "You are a World Cup analyst. Generate a structured match prediction.\n\n"
        "Data:\n{h2h_data}\n\n"
        "Format:\n"
        "## Match Preview\n"
        "**Teams:** [Team A] vs [Team B]\n"
        "**Historical Edge:** [advantage]\n"
        "**Key Stats:** [2-3 stats]\n"
        "**Recent Form:** [analysis]\n"
        "**Prediction:** [score + reasoning]\n"
        "**Confidence:** [Low/Medium/High + why]\n"
        "**Limitations:** [data caveats]\n"
    )
    chain = prompt | llm
    return chain.invoke({"h2h_data": h2h_data}).content

# Collect tools
tools = [
    dataset_discovery_tool,
    data_ingestion_tool,
    retrieval_tool,
    reasoning_tool,
    llm_synthesis_tool,
    report_generation_tool,
]

print("✅ All 6 tools defined:")
for t in tools:
    print(f"   🔧 {t.name}: {t.description[:60]}...")

✅ All 6 tools defined:
   🔧 dataset_discovery_tool: Describes the available World Cup dataset: columns, date ran...
   🔧 data_ingestion_tool: Returns summary statistics about the ingested World Cup data...
   🔧 retrieval_tool: Searches the World Cup vector database for matches relevant ...
   🔧 reasoning_tool: Analyzes head-to-head record between two teams. Computes win...
   🔧 llm_synthesis_tool: Synthesizes a natural language answer using Gemini from retr...
   🔧 report_generation_tool: Generates a structured match prediction report for two teams...


In [ ]:
# CELL 8: Build LangChain Agent with Memory
# ============================================================

memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

AGENT_PROMPT = PromptTemplate.from_template("""You are a World Cup AI Analyst with access to historical football data.

You have these tools:
{tools}

Tool names: {tool_names}

Use this EXACT format for every step:

Question: the user's input
Thought: think about which tool to use
Action: the tool name (must be one of [{tool_names}])
Action Input: your input to the tool
Observation: the tool result
... (repeat Thought/Action/Observation as needed)
Thought: I now know the final answer
Final Answer: your comprehensive response

Guidelines:
- Use retrieval_tool for historical match questions
- Use reasoning_tool for head-to-head comparisons between two teams
- Use report_generation_tool for match predictions
- Always cite specific data from tool results
- State limitations when data is incomplete
- Remember user preferences from chat history

Previous conversation:
{chat_history}

Question: {input}
{agent_scratchpad}""")

agent = create_react_agent(llm, tools, AGENT_PROMPT)
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    memory=memory,
    verbose=True,          # Set False for cleaner output
    handle_parsing_errors=True,
    max_iterations=6,
)

print("✅ Agent ready with memory")

✅ Agent ready with memory


/tmp/ipython-input-2887/4203891763.py:4: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)


In [ ]:
# ============================================================
# CELL 9: Test — Dataset Discovery
# ============================================================
print("=" * 60)
print("TEST 1: What data is available?")
print("=" * 60)
result = agent_executor.invoke({"input": "What World Cup data do you have available?"})
print("\n📝 ANSWER:", result["output"])


TEST 1: What data is available?


> Entering new AgentExecutor chain...
Action: dataset_discovery_tool
Action Input: What World Cup data do you have available?Dataset: International Football Results (World Cup main tournament)
Source: Kaggle (martj42/international-football-results)
Matches: 964
Date range: 1930 to 2022
Columns: ['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'city', 'country', 'neutral', 'result', 'text']
Available teams (82): Algeria, Angola, Argentina, Australia, Austria, Belgium, Bolivia, Bosnia and Herzegovina, Brazil, Bulgaria, Cameroon, Canada, Chile, China PR, Colombia, Costa Rica, Croatia, Cuba, Czech Republic, Czechoslovakia... and more.
Action: data_ingestion_tool
Action Input: summarize ingested World Cup dataIngested 964 World Cup matches.
Average goals per match: 2.82
Most frequent teams:
Brazil         114
Germany        112
Argentina       88
Italy           83
England         74
France          73
Spain           67
Mexico  

In [ ]:
# ============================================================
# CELL 10: Test — Historical RAG Query
# ============================================================
print("=" * 60)
print("TEST 2: Historical Question (RAG)")
print("=" * 60)
result = agent_executor.invoke({"input": "What happened in the 2014 World Cup final?"})
print("\n📝 ANSWER:", result["output"])

TEST 2: Historical Question (RAG)


> Entering new AgentExecutor chain...
Action: retrieval_tool
Action Input: 2014 World Cup final1. On June 16, 2014, in the FIFA World Cup, Germany played against Portugal in Salvador, Brazil. The score was Germany 4 - 0 Portugal. Result: home win.
2. On July 13, 2014, in the FIFA World Cup, Germany played against Argentina in Rio de Janeiro, Brazil. The score was Germany 1 - 0 Argentina. Result: home win.
3. On June 30, 2014, in the FIFA World Cup, Germany played against Algeria in Porto Alegre, Brazil. The score was Germany 2 - 1 Algeria. Result: home win.
4. On June 25, 2014, in the FIFA World Cup, Bosnia and Herzegovina played against Iran in Salvador, Brazil. The score was Bosnia and Herzegovina 3 - 1 Iran. Result: home win.
5. On June 20, 2014, in the FIFA World Cup, Switzerland played against France in Salvador, Brazil. The score was Switzerland 2 - 5 France. Result: away win.
6. On June 21, 2014, in the FIFA World Cup, Germany played against G

In [ ]:
# ============================================================
# CELL 11: Test — Match Prediction
# ============================================================
print("=" * 60)
print("TEST 3: Match Prediction")
print("=" * 60)
result = agent_executor.invoke({
    "input": "Predict the outcome of a World Cup match between Brazil and Germany"
})
print("\n📝 ANSWER:", result["output"])

TEST 3: Match Prediction


> Entering new AgentExecutor chain...
Action: report_generation_tool
Action Input: {"team1": "Brazil", "team2": "Germany"}

* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash
Please retry in 24.208421392s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerMinutePerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.5-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 5
}
, retry_delay {
  seconds: 24
}
].


## Match Preview
**Teams:** Brazil vs Germany
**Historical Edge:** The World Cup head-to-head record between these two giants is perfectly balanced with one win for each nation. However, Germany holds a significant historical edge in terms of goals scored (7-3) due to their dominant 7-1 victory in 2014, a result that continues to cast a long shadow and carries immense psychological weight.
**Key Stats:**
*   World Cup H2H: 1 win for Brazil, 1 win for Germany.
*   H2H Goal Difference: Germany +4 (7 goals scored vs 3 conceded).
*   Brazil's Recent WC Form: 3 wins, 1 draw, 1 loss in their last 5 matches.
**Recent Form:** Brazil enters this potential clash in slightly better recent World Cup form (DWLWW), demonstrating a strong winning mentality and resilience over their last five outings. Germany's form (WDLLW) has been more inconsistent, showing both flashes of brilliance and periods of vulnerability, with two losses in their last five matches.
**Prediction:** Brazil 2 - 1 Germany
**Reas

* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash
Please retry in 22.096396307s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerMinutePerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.5-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 5
}
, retry_delay {
  seconds: 22
}
].
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash
Please retry in 17.992462413s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com

I now know the final answer
Final Answer: ## Match Prediction: Brazil vs Germany

**Historical Context:**
The World Cup head-to-head record between Brazil and Germany is perfectly balanced with one win for each nation. However, Germany holds a significant historical edge in terms of goals scored (7-3) due to their dominant 7-1 victory in 2014, a result that continues to cast a long shadow and carries immense psychological weight.

**Key Stats:**
*   World Cup Head-to-Head: 1 win for Brazil, 1 win for Germany.
*   Head-to-Head Goal Difference: Germany +4 (7 goals scored vs 3 conceded).

**Recent World Cup Form (Last 5 Matches):**
*   **Brazil:** 3 wins, 1 draw, 1 loss (DWLWW)
*   **Germany:** 2 wins, 1 draw, 2 losses (WDLLW)

**Prediction:** Brazil 2 - 1 Germany

**Reasoning:**
Brazil enters this potential clash in slightly better recent World Cup form, demonstrating a strong winning mentality and resilience. The strong, lingering desire for redemption after the traumatic 2014 semi-fina

In [ ]:
# ============================================================
# CELL 12: Test — Memory Persistence (2-turn)
# ============================================================
print("=" * 60)
print("TEST 4a: Setting user preference")
print("=" * 60)
result = agent_executor.invoke({
    "input": "My favorite team is Argentina. How have they done in World Cup finals?"
})
print("\n📝 ANSWER:", result["output"])

print("\n" + "=" * 60)
print("TEST 4b: Follow-up using memory")
print("=" * 60)
result = agent_executor.invoke({
    "input": "Based on my favorite team, predict their chances against France in a World Cup match"
})
print("\n📝 ANSWER:", result["output"])

TEST 4a: Setting user preference


> Entering new AgentExecutor chain...
Action: retrieval_tool
Action Input: Argentina World Cup finals1. On June 15, 2014, in the FIFA World Cup, Argentina played against Bosnia and Herzegovina in Rio de Janeiro, Brazil. The score was Argentina 2 - 1 Bosnia and Herzegovina. Result: home win.
2. On November 26, 2022, in the FIFA World Cup, Argentina played against Mexico in Lusail, Qatar. The score was Argentina 2 - 0 Mexico. Result: home win.
3. On December 13, 2022, in the FIFA World Cup, Argentina played against Croatia in Lusail, Qatar. The score was Argentina 3 - 0 Croatia. Result: home win.
4. On December 03, 2022, in the FIFA World Cup, Argentina played against Australia in Al Rayyan, Qatar. The score was Argentina 2 - 1 Australia. Result: home win.
5. On November 22, 2022, in the FIFA World Cup, Argentina played against Saudi Arabia in Lusail, Qatar. The score was Argentina 1 - 2 Saudi Arabia. Result: away win.
6. On December 18, 2022, in the FI

* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash
Please retry in 7.304602695s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerMinutePerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.5-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 5
}
, retry_delay {
  seconds: 7
}
].


Action: report_generation_tool
Action Input: {"team1": "Argentina", "team2": "France"}

* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash
Please retry in 5.20051288s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerMinutePerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.5-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 5
}
, retry_delay {
  seconds: 5
}
].
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash
Please retry in 1.109585586s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/gen

## Match Preview
**Teams:** Argentina vs France
**Historical Edge:** Argentina holds a slight historical advantage in World Cup encounters, leading the head-to-head record with 2 wins to France's 1, alongside 1 draw. They also have a marginal edge in goals scored (Argentina 9 - 8 France).
**Key Stats:**
*   Argentina leads the World Cup H2H 2-1-1 (W-L-D).
*   The most recent encounter was a thrilling 3-3 draw in the 2022 World Cup Final, showcasing extreme parity.
*   Goals scored across all World Cup meetings are incredibly close: Argentina 9 - 8 France.
**Recent Form:**
*   **Argentina:** DWDWW (Unbeaten in their last 4 World Cup matches, with 3 wins and 1 draw). This indicates strong, consistent form.
*   **France:** DWWWL (3 wins, 1 draw, and 1 loss in their last 5 World Cup matches). While generally strong, the recent loss suggests a slight vulnerability compared to Argentina's unbeaten streak. The 'D' is notably the 2022 final against Argentina.
**Prediction:** Argentina 2-2 Fran

In [ ]:
# CELL 13: Visualization — Team World Cup History
# ============================================================

def plot_team_wc_history(team_name):
    """Plotly chart of a team's World Cup goals by tournament year."""
    tm = wc_main[(wc_main["home_team"] == team_name) | (wc_main["away_team"] == team_name)].copy()
    if tm.empty:
        print(f"No World Cup data for {team_name}")
        return

    tm["goals_for"] = tm.apply(
        lambda r: r["home_score"] if r["home_team"] == team_name else r["away_score"], axis=1
    )
    tm["goals_against"] = tm.apply(
        lambda r: r["away_score"] if r["home_team"] == team_name else r["home_score"], axis=1
    )
    tm["year"] = tm["date"].dt.year
    yr = tm.groupby("year").agg(
        matches=("date", "count"),
        goals_for=("goals_for", "sum"),
        goals_against=("goals_against", "sum"),
    ).reset_index()

    fig = go.Figure()
    fig.add_trace(go.Bar(x=yr["year"], y=yr["goals_for"], name="Goals For", marker_color="#2ecc71"))
    fig.add_trace(go.Bar(x=yr["year"], y=yr["goals_against"], name="Goals Against", marker_color="#e74c3c"))
    fig.update_layout(
        title=f"⚽ {team_name} — World Cup Goal History",
        xaxis_title="Year", yaxis_title="Goals",
        barmode="group", template="plotly_dark",
        width=900, height=450,
    )
    fig.show()

# Generate charts for key teams
plot_team_wc_history("Brazil")
plot_team_wc_history("Argentina")
plot_team_wc_history("Germany")



In [ ]:
# ============================================================
# CELL 14: Interactive Chat Loop (Optional)
# ============================================================

def chat(question):
    """Helper to chat with the agent."""
    result = agent_executor.invoke({"input": question})
    print(f"\n🤖 {result['output']}")
    return result["output"]

# Uncomment to use interactively:
# chat("Who won the most World Cups?")
# chat("Compare France and Brazil head to head")
# chat("Predict Argentina vs Germany")

print("\n✅ Notebook complete!")



✅ Notebook complete!


In [ ]:
!pip install python-pptx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.8/472.8 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 14.3 MB/s eta 0:00:00


In [ ]:
# ============================================================
# CELL 15: Install Streamlit + Tunnel
# ============================================================
!pip install -q streamlit



In [ ]:
# ============================================================
# CELL 16: Write app.py to disk
# ============================================================
%%writefile app.py
import os
import streamlit as st
import pandas as pd
import plotly.graph_objects as go
from pathlib import Path

from langchain_core.tools import tool
from langchain_core.prompts import PromptTemplate
from langchain_core.documents import Document
from langchain.agents import AgentExecutor, create_react_agent
from langchain.memory import ConversationBufferMemory
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

# ── Page Config ─────────────────────────────────────────────
st.set_page_config(page_title="⚽ World Cup AI Analyst", page_icon="⚽", layout="wide")

st.markdown("""
<style>
    .stApp {
        background: linear-gradient(135deg, #0f0c29, #302b63, #24243e);
    }
    h1 { color: #f1c40f !important; }
    .stChatMessage { border-radius: 12px; }
</style>
""", unsafe_allow_html=True)

st.title("⚽ World Cup Match Predictor & Analyst")
st.caption("Powered by LangChain + Gemini 2.5 Flash | Ask about World Cup history, stats & predictions")

# ── Sidebar ─────────────────────────────────────────────────
with st.sidebar:
    st.header("⚙️ Settings")
    api_key = st.text_input("Gemini API Key", type="password",
                            value=os.environ.get("GOOGLE_API_KEY", ""))
    if api_key:
        os.environ["GOOGLE_API_KEY"] = api_key

    st.divider()
    st.header("📊 Quick Actions")
    quick = st.radio("Try a quick query:", [
        "None",
        "What data is available?",
        "2014 World Cup Final",
        "Predict: Brazil vs Germany",
        "Predict: Argentina vs France",
    ])

    st.divider()
    st.header("🏟️ Team Visualizer")
    team_input = st.text_input("Team name for chart", placeholder="e.g. Brazil")

# ── Load Data ───────────────────────────────────────────────
@st.cache_data
def load_data():
    fp = Path("data/results.csv")
    if not fp.exists():
        st.error("❌ Place results.csv in the data/ folder.")
        st.stop()
    df = pd.read_csv(fp, parse_dates=["date"])
    wc = df[df["tournament"].str.contains("FIFA World Cup", case=False, na=False)].copy()
    wc = wc[~wc["tournament"].str.contains("qualification", case=False, na=False)].copy()

    def res(r):
        if r["home_score"] > r["away_score"]: return "home_win"
        elif r["home_score"] < r["away_score"]: return "away_win"
        return "draw"

    wc["result"] = wc.apply(res, axis=1)
    wc["text"] = wc.apply(lambda r: (
        f"On {r['date'].strftime('%B %d, %Y')}, in the {r['tournament']}, "
        f"{r['home_team']} played {r['away_team']} in {r['city']}, {r['country']}. "
        f"Score: {r['home_team']} {r['home_score']} - {r['away_score']} {r['away_team']}. "
        f"Result: {r['result'].replace('_', ' ')}."
    ), axis=1)
    return wc

wc_main = load_data()

# ── Vector Store ────────────────────────────────────────────
@st.cache_resource
def build_vectorstore(_wc):
    emb = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    idx = Path("cache/faiss_index")
    if (idx / "index.faiss").exists():
        return FAISS.load_local(str(idx), emb, allow_dangerous_deserialization=True)
    docs = [
        Document(page_content=r["text"], metadata={
            "date": str(r["date"].date()), "home_team": r["home_team"],
            "away_team": r["away_team"], "home_score": int(r["home_score"]),
            "away_score": int(r["away_score"]), "tournament": r["tournament"],
        }) for _, r in _wc.iterrows()
    ]
    vs = FAISS.from_documents(docs, emb)
    idx.mkdir(parents=True, exist_ok=True)
    vs.save_local(str(idx))
    return vs

vectorstore = build_vectorstore(wc_main)
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

# ── Define Tools ────────────────────────────────────────────
@tool
def dataset_discovery_tool(query: str) -> str:
    """Describes the available World Cup dataset."""
    teams = sorted(set(wc_main["home_team"]) | set(wc_main["away_team"]))
    return (f"Dataset: World Cup matches (main tournament)\nMatches: {len(wc_main)}\n"
            f"Range: {wc_main['date'].min().year}-{wc_main['date'].max().year}\n"
            f"Teams ({len(teams)}): {', '.join(teams[:15])}...")

@tool
def data_ingestion_tool(query: str) -> str:
    """Returns summary stats of the World Cup dataset."""
    apps = (wc_main["home_team"].value_counts() + wc_main["away_team"].value_counts()).fillna(0).astype(int)
    avg = wc_main[["home_score","away_score"]].sum().sum() / len(wc_main)
    return f"Matches: {len(wc_main)}\nAvg goals/match: {avg:.2f}\nTop teams:\n{apps.sort_values(ascending=False).head(10).to_string()}"

@tool
def retrieval_tool(query: str) -> str:
    """Searches the World Cup vector database for relevant historical matches."""
    docs = retriever.invoke(query)
    if not docs: return "No matches found."
    return "\n".join(f"{i}. {d.page_content}" for i, d in enumerate(docs, 1))

@tool
def reasoning_tool(query: str) -> str:
    """Computes head-to-head stats between two teams in World Cup history."""
    all_teams = set(wc_main["home_team"]) | set(wc_main["away_team"])
    found = [t for t in all_teams if t.lower() in query.lower()]
    if len(found) < 2:
        return f"Need two team names. Found: {found}"
    t1, t2 = found[0], found[1]
    h2h = wc_main[
        ((wc_main["home_team"]==t1)&(wc_main["away_team"]==t2)) |
        ((wc_main["home_team"]==t2)&(wc_main["away_team"]==t1))
    ].sort_values("date")
    if h2h.empty: return f"No WC matches between {t1} and {t2}."
    w1=w2=d=g1=g2=0
    details=[]
    for _,r in h2h.iterrows():
        a,b = (r["home_score"],r["away_score"]) if r["home_team"]==t1 else (r["away_score"],r["home_score"])
        g1+=a; g2+=b
        if a>b: w1+=1
        elif b>a: w2+=1
        else: d+=1
        details.append(f"  {r['date'].strftime('%Y-%m-%d')}: {r['home_team']} {r['home_score']}-{r['away_score']} {r['away_team']}")

    def form(t, n=5):
        m = wc_main[(wc_main["home_team"]==t)|(wc_main["away_team"]==t)].sort_values("date",ascending=False).head(n)
        return "".join(
            "W" if (r["home_score"]>r["away_score"] if r["home_team"]==t else r["away_score"]>r["home_score"])
            else "L" if (r["home_score"]<r["away_score"] if r["home_team"]==t else r["away_score"]<r["home_score"])
            else "D" for _,r in m.iterrows()
        )

    return (f"H2H: {t1} vs {t2}\nMatches: {len(h2h)}\n{t1} wins: {w1} | {t2} wins: {w2} | Draws: {d}\n"
            f"Goals: {t1} {g1}-{g2} {t2}\nForm(last 5): {t1}:{form(t1)} {t2}:{form(t2)}\n" + "\n".join(details))

@tool
def llm_synthesis_tool(query: str) -> str:
    """Synthesizes a natural language answer from data context using Gemini."""
    p = PromptTemplate.from_template(
        "You are a World Cup expert. Answer based on:\n{query}\nInclude key facts and limitations.")
    return (p | llm).invoke({"query": query}).content

@tool
def report_generation_tool(query: str) -> str:
    """Generates a structured match prediction report for two teams."""
    h2h = reasoning_tool.invoke(query)
    p = PromptTemplate.from_template(
        "You are a World Cup analyst. Generate a match prediction.\nData:\n{h2h}\n\n"
        "Format: ## Match Preview\n**Teams:** ...\n**Historical Edge:** ...\n"
        "**Key Stats:** ...\n**Prediction:** ...\n**Confidence:** ...\n**Limitations:** ...")
    return (p | llm).invoke({"h2h": h2h}).content

tools = [dataset_discovery_tool, data_ingestion_tool, retrieval_tool,
         reasoning_tool, llm_synthesis_tool, report_generation_tool]

# ── Agent ───────────────────────────────────────────────────
@st.cache_resource
def get_llm():
    return ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.3, max_output_tokens=2048)

llm = get_llm()

if "memory" not in st.session_state:
    st.session_state.memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

AGENT_PROMPT = PromptTemplate.from_template(
    """You are a World Cup AI Analyst. Tools:\n{tools}\nTool names: {tool_names}\n
Use this format:
Question: the question
Thought: which tool to use
Action: tool name
Action Input: input
Observation: result
... (repeat as needed)
Thought: I now know the answer
Final Answer: your response

Rules: Use retrieval_tool for history, reasoning_tool for H2H, report_generation_tool for predictions.
Cite data. State limitations. Remember user preferences.

Chat history:\n{chat_history}\n
Question: {input}\n{agent_scratchpad}""")

agent = create_react_agent(llm, tools, AGENT_PROMPT)
executor = AgentExecutor(
    agent=agent, tools=tools, memory=st.session_state.memory,
    verbose=False, handle_parsing_errors=True, max_iterations=6)

# ── Chat UI ─────────────────────────────────────────────────
if "messages" not in st.session_state:
    st.session_state.messages = [
        {"role": "assistant", "content": "⚽ Welcome! Ask me about World Cup history, team stats, or match predictions.\n\nTry: *Predict Brazil vs Germany*"}
    ]

for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

# Handle quick action or typed input
prompt = None
if quick and quick != "None":
    prompt = quick
else:
    prompt = st.chat_input("Ask about World Cup history or predictions...")

if prompt:
    if not os.environ.get("GOOGLE_API_KEY"):
        st.error("Please enter your Gemini API key in the sidebar.")
        st.stop()

    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)

    with st.chat_message("assistant"):
        with st.spinner("⚽ Analyzing World Cup data..."):
            try:
                result = executor.invoke({"input": prompt})
                response = result["output"]
            except Exception as e:
                response = f"⚠️ Error: {str(e)}\n\nTry rephrasing your question."
            st.markdown(response)
    st.session_state.messages.append({"role": "assistant", "content": response})

# ── Team Chart in Sidebar ───────────────────────────────────
if team_input:
    tm = wc_main[(wc_main["home_team"]==team_input)|(wc_main["away_team"]==team_input)].copy()
    if tm.empty:
        st.sidebar.warning(f"No WC data for '{team_input}'")
    else:
        tm["gf"] = tm.apply(lambda r: r["home_score"] if r["home_team"]==team_input else r["away_score"], axis=1)
        tm["ga"] = tm.apply(lambda r: r["away_score"] if r["home_team"]==team_input else r["home_score"], axis=1)
        tm["year"] = tm["date"].dt.year
        yr = tm.groupby("year").agg(gf=("gf","sum"), ga=("ga","sum")).reset_index()
        fig = go.Figure()
        fig.add_trace(go.Bar(x=yr["year"], y=yr["gf"], name="Goals For", marker_color="#2ecc71"))
        fig.add_trace(go.Bar(x=yr["year"], y=yr["ga"], name="Goals Against", marker_color="#e74c3c"))
        fig.update_layout(title=f"{team_input} WC Goals", barmode="group",
                          template="plotly_dark", height=300)
        st.sidebar.plotly_chart(fig, use_container_width=True)

# ── Footer ──────────────────────────────────────────────────
st.sidebar.divider()
st.sidebar.markdown("**Data:** [Kaggle — Intl Football Results](https://www.kaggle.com/datasets/martj42/international-football-results-from-1872-to-2017)")
st.sidebar.markdown("**Stack:** LangChain + Gemini 2.5 Flash + FAISS + Streamlit")




Overwriting app.py


In [ ]:
# ============================================================
# CELL 17: Launch Streamlit with Tunnel
# ============================================================
# This creates a public URL you can open in a new tab

!npx localtunnel --port 8501 &
!streamlit run app.py --server.port 8501 &

import time
time.sleep(5)

from IPython.display import display, HTML
display(HTML("""
<h3>🚀 Streamlit App is Running!</h3>
<p><b>Steps:</b></p>
<ol>
    <li>Run this cell — wait for the tunnel URL to appear below</li>
    <li>Click the <b>localtunnel URL</b> (looks like https://xxxxx.loca.lt)</li>
    <li>On the tunnel page, enter the <b>IP address</b> shown and click Submit</li>
    <li>Your Streamlit app will load! Enter your Gemini API key in the sidebar.</li>
</ol>
"""))

# Get your Colab IP (needed for the tunnel password page)
!wget -q -O - ipv4.icanhazip.com

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸your url is: https://blue-mails-lead.loca.lt



  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.106.48.119:8501

